# Video Game Sales Prediction Using Machine Learning

**Academic Machine Learning Project**

This project analyzes video game sales data and predicts global sales using three regression algorithms: Linear Regression, Decision Tree Regressor, and Random Forest Regressor.

The notebook includes dataset inspection, cleaning, exploratory data analysis, preprocessing, model training, evaluation, cross-validation, hyperparameter tuning, sales-category confusion matrices, and saving the best model.

**Dataset required:** `vgsales.csv` (upload it when the notebook asks for a file).

**Import the Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_validate, KFold
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully!")

***Dataset Upload***

In [ ]:
import pandas as pd

df = pd.read_csv("vgsales.csv")

print("Dataset Loaded Successfully!")
print("Dataset Shape:", df.shape)

display(df.head())

**Dataset Load**

In [ ]:
df = pd.read_csv("vgsales.csv")

print("Dataset loaded successfully!")


**Print First Five Rows**

In [ ]:
df.head()

**Print Last Five Rows**

In [ ]:
df.tail()

**Total Numbers of Rows and Columns**

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

**Print the name of Columns**

In [ ]:
print(df.columns.tolist())

**Check Datatypes**

In [ ]:
df.info()

**Check Statistical Summary**

In [ ]:
df.describe()

**Statistical Summary for all the Columns**

In [ ]:
df.describe(include="all")

**Check Missing Values**

In [ ]:
print("Missing values in each column:")

print(df.isnull().sum())

**Check Duplicate Records**

In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())

**Check Missing Values Percentage**

In [ ]:
missing_percentage = (
    df.isnull().sum() / len(df) * 100
).sort_values(ascending=False)

print(missing_percentage)

**Clean the Dataset**

In [ ]:

df_clean = df.copy()

df_clean = df_clean.drop_duplicates()


df_clean["Year"] = pd.to_numeric(
    df_clean["Year"], errors="coerce"
)

df_clean["Global_Sales"] = pd.to_numeric(
    df_clean["Global_Sales"], errors="coerce"
)


df_clean = df_clean.dropna(
    subset=["Global_Sales"]
)

df_clean = df_clean[
    df_clean["Global_Sales"] >= 0
]

df_clean.loc[
    df_clean["Year"].between(1950, 2035),
    "Year"
] = np.nan

print("Original dataset shape:", df.shape)
print("Cleaned dataset shape:", df_clean.shape)

df_clean.head()

**Saving Cleaned Dataset**

In [ ]:
df_clean.to_csv(
    "vgsales_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

**Check Year and Global Sales Data**

In [ ]:
print("Dataset shape:", df.shape)

print("\nYear column:")
print(df["Year"].head(10))

print("\nValid Year count:", df["Year"].notna().sum())

print("\nGlobal Sales column:")
print(df["Global_Sales"].head(10))

print("\nValid Global Sales count:", df["Global_Sales"].notna().sum())

**Year Wise Sales**

In [ ]:

df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Global_Sales"] = pd.to_numeric(
    df["Global_Sales"], errors="coerce"
)

year_data = df.dropna(subset=["Year", "Global_Sales"]).copy()

year_sales = year_data.groupby("Year")["Global_Sales"].sum().sort_index()

plt.figure(figsize=(12, 5))

plt.plot(
    year_sales.index,
    year_sales.values,
    marker="o"
)

plt.xlabel("Release Year")
plt.ylabel("Total Global Sales (Millions)")
plt.title("Global Video Game Sales by Year")
plt.grid(True)
plt.tight_layout()
plt.show()

**EDA of Global Sales Distribution**

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    df_clean["Global_Sales"],
    bins=50,
    edgecolor="black"
)

plt.xlabel("Global Sales (Millions)")
plt.ylabel("Number of Games")
plt.title("Distribution of Global Video Game Sales")

plt.show()

**Genre Wise Sales**

In [ ]:
genre_sales = (
    df_clean.groupby("Genre")["Global_Sales"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 5))

genre_sales.plot(kind="bar")

plt.xlabel("Genre")
plt.ylabel("Total Global Sales (Millions)")
plt.title("Total Global Sales by Genre")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

print(genre_sales)

**Platform Wise Sales**

In [ ]:
platform_sales = (
    df_clean.groupby("Platform")["Global_Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(10, 5))

platform_sales.plot(kind="bar")

plt.xlabel("Platform")
plt.ylabel("Total Global Sales (Millions)")
plt.title("Top 15 Platforms by Global Sales")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

**Select input features and target**

**Input features (X)
Platform
Genre
Year
Publisher**
**Target variable (y):**
**Global_Sales**

In [ ]:
features = [
    "Platform",
    "Genre",
    "Year",
    "Publisher"
]

target = "Global_Sales"

X = df_clean[features].copy()

y = df_clean[target].copy()

print("Input features:")
print(X.head())

print("\nTarget variable:")
print(y.head())

**Encode Categorical data and Handle Missing Values**

In [ ]:


# Define input features
numerical_features = ["Year"]

categorical_features = ["Platform", "Genre", "Publisher"]

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Preprocessing pipeline created successfully!")

**Split the Dataset into Training and Testing Data**

In [ ]:
from sklearn.model_selection import train_test_split

X = df[["Platform", "Genre", "Year", "Publisher"]]
y = df["Global_Sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

**Train the Linear Regression Model and Evaluate**

In [ ]:
linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

# Train the model
linear_model.fit(X_train, y_train)

# Predict sales
y_pred_linear = linear_model.predict(X_test)

print("Linear Regression training completed!")
mae_linear = mean_absolute_error(
    y_test,
    y_pred_linear
)

mse_linear = mean_squared_error(
    y_test,
    y_pred_linear
)

rmse_linear = np.sqrt(mse_linear)

r2_linear = r2_score(
    y_test,
    y_pred_linear
)

print("Linear Regression Results")
print("--------------------------")
print("MAE:", mae_linear)
print("MSE:", mse_linear)
print("RMSE:", rmse_linear)
print("R2 Score:", r2_linear)

**Train the Decision Tree Regressor and Evaluate**

In [ ]:
decision_tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeRegressor(
                random_state=42,
                min_samples_leaf=2
            )
        )
    ]
)

# Train model
decision_tree_model.fit(
    X_train,
    y_train
)

# Predict sales
y_pred_tree = decision_tree_model.predict(
    X_test
)

print("Decision Tree training completed!")
mae_tree = mean_absolute_error(
    y_test,
    y_pred_tree
)

mse_tree = mean_squared_error(
    y_test,
    y_pred_tree
)

rmse_tree = np.sqrt(mse_tree)

r2_tree = r2_score(
    y_test,
    y_pred_tree
)

print("Decision Tree Results")
print("----------------------")
print("MAE:", mae_tree)
print("MSE:", mse_tree)
print("RMSE:", rmse_tree)
print("R2 Score:", r2_tree)

**Train the Random Forest Regressor and Evaluate**

In [ ]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=150,
                random_state=42,
                n_jobs=-1,
                min_samples_leaf=2
            )
        )
    ]
)

# Train model
random_forest_model.fit(
    X_train,
    y_train
)

# Predict sales
y_pred_rf = random_forest_model.predict(
    X_test
)

print("Random Forest training completed!")
mae_rf = mean_absolute_error(
    y_test,
    y_pred_rf
)

mse_rf = mean_squared_error(
    y_test,
    y_pred_rf
)

rmse_rf = np.sqrt(mse_rf)

r2_rf = r2_score(
    y_test,
    y_pred_rf
)

print("Random Forest Results")
print("---------------------")
print("MAE:", mae_rf)
print("MSE:", mse_rf)
print("RMSE:", rmse_rf)
print("R2 Score:", r2_rf)

**Compare the Three Models**

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree Regressor",
        "Random Forest Regressor"
    ],

    "MAE": [
        mae_linear,
        mae_tree,
        mae_rf
    ],

    "MSE": [
        mse_linear,
        mse_tree,
        mse_rf
    ],

    "RMSE": [
        rmse_linear,
        rmse_tree,
        rmse_rf
    ],

    "R2 Score": [
        r2_linear,
        r2_tree,
        r2_rf
    ]
})

results = results.sort_values(
    by="MAE"
).reset_index(drop=True)

results

**Perform 5-Fold Cross-Validation**

**Define 5-Fold Cross-Validation**

In [ ]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-Fold Cross-Validation initialized!")

**Define the Three Models**

In [ ]:
models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree Regressor": DecisionTreeRegressor(
        random_state=42,
        min_samples_leaf=2
    ),

    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=150,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    )
}

**Run Cross-Validation**

In [ ]:
cv_results = []

for name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "MSE": "neg_mean_squared_error",
            "R2": "r2"
        },
        n_jobs=1
    )

    # Convert negative error scores into positive values
    mae_scores = -scores["test_MAE"]
    mse_scores = -scores["test_MSE"]
    rmse_scores = np.sqrt(mse_scores)
    r2_scores = scores["test_R2"]

    cv_results.append({
        "Model": name,
        "Mean CV MAE": mae_scores.mean(),
        "Std CV MAE": mae_scores.std(ddof=1),
        "Mean CV MSE": mse_scores.mean(),
        "Mean CV RMSE": rmse_scores.mean(),
        "Mean CV R2": r2_scores.mean()
    })

    print(f"{name} completed!")

**Display the Cross-Validation Results**

In [ ]:
cv_results_df = pd.DataFrame(cv_results)

cv_results_df = cv_results_df.sort_values(
    by="Mean CV MAE"
).reset_index(drop=True)

cv_results_df

**Save the Cross-Validation Results**

In [ ]:
cv_results_df.to_csv(
    "cross_validation_results.csv",
    index=False
)

print("Cross-validation results saved successfully!")

**Hyperparameter Tuning**

**Decision Tree Hyperparameter Tuning**

In [ ]:
# Create Decision Tree Pipeline

dt_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42))
])

# Define parameters

dt_params = {
    "model__max_depth": [5, 10, 20, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

# Apply Grid Search

dt_grid = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=dt_params,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)

# Train

dt_grid.fit(X_train, y_train)

print("Best Decision Tree Parameters:")
print(dt_grid.best_params_)

print("\nBest Cross Validation MAE:")
print(-dt_grid.best_score_)

**Random Forest Hyperparameter Tuning**

In [ ]:
# Create Random Forest Pipeline

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        random_state=42,
        n_jobs=1
    ))
])

# Define parameters

rf_params = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [20, None],
    "model__min_samples_leaf": [1, 2]
}

# Apply Grid Search

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_params,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)

# Train

rf_grid.fit(X_train, y_train)

print("Best Random Forest Parameters:")
print(rf_grid.best_params_)

print("\nBest Cross Validation MAE:")
print(-rf_grid.best_score_)

**Hyperparameter Tuning Comparison**

In [ ]:
tuning_results = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Random Forest"
    ],
    "Best CV MAE": [
        -dt_grid.best_score_,
        -rf_grid.best_score_
    ]
})

print(tuning_results)

**Prepare Model for Confusion Matrix**

In [ ]:
# Linear Regression
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

# Tuned models
best_dt_model = dt_grid.best_estimator_
best_rf_model = rf_grid.best_estimator_

print("All models are ready!")

**Generate Sales Predictions**

In [ ]:
# Predictions on test data

linear_predictions = linear_model.predict(X_test)

dt_predictions = best_dt_model.predict(X_test)

rf_predictions = best_rf_model.predict(X_test)

print("Predictions generated successfully!")

**Convert Sales into Low and High Categories**

In [ ]:
# Threshold = 1 million copies

def convert_sales_category(sales):
    return np.where(sales < 1, "Low Sales", "High Sales")


# Actual categories

actual_categories = convert_sales_category(y_test)

# Predicted categories

linear_categories = convert_sales_category(linear_predictions)

dt_categories = convert_sales_category(dt_predictions)

rf_categories = convert_sales_category(rf_predictions)

print("Sales categories generated successfully!")

**Generate Confusion Matrices**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

models_categories = {
    "Linear Regression": linear_categories,
    "Decision Tree": dt_categories,
    "Random Forest": rf_categories
}

for model_name, predictions in models_categories.items():

    cm = confusion_matrix(
        actual_categories,
        predictions,
        labels=["Low Sales", "High Sales"]
    )

    plt.figure(figsize=(6, 4))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Low Sales", "High Sales"],
        yticklabels=["Low Sales", "High Sales"]
    )

    plt.xlabel("Predicted Category")
    plt.ylabel("Actual Category")
    plt.title(model_name + " Confusion Matrix")

    plt.tight_layout()
    plt.show()

    print("\n", model_name)
    print(classification_report(
        actual_categories,
        predictions,
        zero_division=0
    ))

**Save Confusion Matrices**

In [ ]:
import os

os.makedirs("project_results", exist_ok=True)

for model_name, predictions in models_categories.items():

    cm = confusion_matrix(
        actual_categories,
        predictions,
        labels=["Low Sales", "High Sales"]
    )

    cm_df = pd.DataFrame(
        cm,
        index=["Actual Low Sales", "Actual High Sales"],
        columns=["Predicted Low Sales", "Predicted High Sales"]
    )

    filename = model_name.lower().replace(" ", "_")

    cm_df.to_csv(
        f"project_results/{filename}_confusion_matrix.csv"
    )

print("All confusion matrices saved successfully!")

**Final Model Evaluation**

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

final_models = {
    "Linear Regression": linear_model,
    "Tuned Decision Tree": best_dt_model,
    "Tuned Random Forest": best_rf_model
}

evaluation_results = []

for name, model in final_models.items():

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)

    mse = mean_squared_error(y_test, predictions)

    rmse = np.sqrt(mse)

    r2 = r2_score(y_test, predictions)

    evaluation_results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    })

final_results = pd.DataFrame(evaluation_results)

print(final_results)

**Final Model Performance Graph**

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    final_results["Model"],
    final_results["MAE"]
)

plt.xlabel("Machine Learning Models")
plt.ylabel("Mean Absolute Error")

plt.title("Final Model Comparison - MAE")

plt.xticks(rotation=15)

plt.grid(axis="y")

plt.tight_layout()
plt.show()

**Select Best Model**

In [ ]:
from sklearn.model_selection import cross_val_score

cv_selection_results = []

for name, model in final_models.items():

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    cv_mae = -scores.mean()

    cv_std = scores.std()

    cv_selection_results.append({
        "Model": name,
        "Mean CV MAE": cv_mae,
        "CV Standard Deviation": cv_std
    })

cv_selection_df = pd.DataFrame(cv_selection_results)

print(cv_selection_df)

best_model_name = cv_selection_df.loc[
    cv_selection_df["Mean CV MAE"].idxmin(),
    "Model"
]

best_model = final_models[best_model_name]

print("\nBest Selected Model:", best_model_name)

**Save Final Machine Learning Model**

In [ ]:
import joblib

# Save complete pipeline

joblib.dump(
    best_model,
    "project_results/best_video_game_sales_model.pkl"
)

print("Final model saved successfully!")

**Save Final Evaluation Results**

In [ ]:
# Save model comparison

final_results.to_csv(
    "project_results/final_model_comparison.csv",
    index=False
)

# Save CV comparison

cv_selection_df.to_csv(
    "project_results/final_cross_validation_results.csv",
    index=False
)

# Save tuning results

tuning_results.to_csv(
    "project_results/hyperparameter_tuning_results.csv",
    index=False
)

print("All evaluation results saved successfully!")

**Download Project Results**

In [ ]:
import shutil

shutil.make_archive(
    "Video_Game_Sales_Project_Results",
    "zip",
    "project_results"
)

from google.colab import files

files.download("Video_Game_Sales_Project_Results.zip")